# Notebook 26 - Recovery-seed factorial

Single self-contained cell. Frozen 40% structures from the 17b/20b registries; no new selection, no test access. 5 methods x 5 seeds x 2 regimes per architecture. Set RUN_ARCHITECTURES in stage 1 to ['shallow'] or ['deep'] for staged runs. Resumable per method-seed cell.

In [ ]:
# ===== stage 0 =====
# Colab/repository bootstrap
from pathlib import Path
import os, sys, json, subprocess, platform, hashlib, random

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

candidates = [
    os.environ.get("SABER_REPO"),
    "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression",
    "/content/localrepo",
    str(Path.cwd()),
]
REPO = None
for candidate in candidates:
    if not candidate:
        continue
    p = Path(candidate).expanduser()
    if (p / "src" / "saber").is_dir() and (p / "config" / "saber.yaml").exists():
        REPO = p.resolve()
        break
if REPO is None:
    raise FileNotFoundError("Repository not found. Set SABER_REPO or edit the candidate path.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repository:", REPO)
print("Python:", sys.version.split()[0], "| Platform:", platform.platform())

# ===== stage 1 =====
# Imports and frozen experiment configuration
from copy import deepcopy
import datetime
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
import yaml
from scipy.stats import spearmanr, t as student_t
import matplotlib.pyplot as plt

from src.saber.bridge_ciciot import load_bridge
from src.saber.deep_model import DeepCNN1D
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import (
    full_model_audit,
    action_weighted_boundary_inversion_rate,
)
from src.saber.adapters import collect_logits, file_sha256, config_hash
from src.saber.surgery import prune_cnn1d_channels, profile_forward_flops, count_parameters

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open(REPO / "config" / "saber.yaml", "r", encoding="utf-8") as handle:
    CFG = yaml.safe_load(handle)

OUT = REPO / "results" / "saber" / "25_recovery_seed_factorial"
OUT.mkdir(parents=True, exist_ok=True)

TARGET_FLOPS = 0.40
METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
RECOVERY_SEEDS = [101, 211, 307, 401, 503]
RUN_ARCHITECTURES = ["shallow", "deep"]  # may be reduced for staged Colab execution
SAVE_CHECKPOINTS = False

PROTOCOLS = {
    "shallow": ["minimal_ce", "full_ce"],
    "deep": ["minimal_ce", "standard_ce"],
}

MINIMUM_WIDTH = int(CFG["groups"]["minimum_remaining_per_layer"])
print("Device:", DEVICE)
print("Methods:", METHODS)
print("Seeds:", RECOVERY_SEEDS)
print("Protocols:", PROTOCOLS)

# ===== stage 2 =====
TRAIN_LOADER, VAL_LOADER, _TEST_LOADER_UNUSED, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
SHALLOW_TEACHER = SHALLOW_TEACHER.to(DEVICE).eval()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)

GRAPH_PATH = REPO / "results/saber/14_risk_graph/asvg_edges_robust.csv"
SHALLOW_REGISTRY_PATH = REPO / "results/saber/17b_calibrated_checkpoint_freeze/shallow_frozen_model_registry.csv"
DEEP_REGISTRY_PATH = REPO / "results/saber/20b_depth_checkpoint_freeze/deep_frozen_model_registry.csv"
DEEP_TEACHER_PATH = REPO / "models/ciciot2023/deepcnn1d_g5_seed0.pt"

for required in [GRAPH_PATH, SHALLOW_REGISTRY_PATH, DEEP_REGISTRY_PATH, DEEP_TEACHER_PATH]:
    if not required.exists():
        raise FileNotFoundError(f"Required frozen artifact is missing: {required}")

robust_graph = pd.read_csv(GRAPH_PATH)
shallow_registry = pd.read_csv(SHALLOW_REGISTRY_PATH)
deep_registry = pd.read_csv(DEEP_REGISTRY_PATH)

# Verify checkpoint registries even though raw models are reconstructed from removed-group files.
for registry, name in [(shallow_registry, "shallow"), (deep_registry, "deep")]:
    for row in registry.itertuples():
        checkpoint = REPO / str(row.checkpoint)
        if not checkpoint.exists() or file_sha256(checkpoint) != str(row.checkpoint_sha256):
            raise RuntimeError(f"{name} registry checkpoint integrity failure: {checkpoint}")

# Deep teacher.
payload = torch.load(DEEP_TEACHER_PATH, map_location="cpu", weights_only=False)
DEEP_TEACHER = DeepCNN1D(len(CLASS_NAMES))
DEEP_TEACHER.load_state_dict(payload["state_dict"])
DEEP_TEACHER = DEEP_TEACHER.to(DEVICE).eval()

# Teacher validation outputs are computed once under the same loader.
SHALLOW_TEACHER_LOGITS, VAL_LABELS, _ = collect_logits(SHALLOW_TEACHER, VAL_LOADER, device=DEVICE)
DEEP_TEACHER_LOGITS, deep_labels, _ = collect_logits(DEEP_TEACHER, VAL_LOADER, device=DEVICE)
if not np.array_equal(VAL_LABELS, deep_labels):
    raise RuntimeError("Shallow/deep validation labels are not aligned.")

TEACHERS = {"shallow": SHALLOW_TEACHER, "deep": DEEP_TEACHER}
TEACHER_LOGITS = {"shallow": SHALLOW_TEACHER_LOGITS, "deep": DEEP_TEACHER_LOGITS}

example = next(iter(VAL_LOADER))[0][:8].float().to(DEVICE)
print("Validation n:", len(VAL_LABELS))
print("Shallow registry rows:", len(shallow_registry))
print("Deep registry rows:", len(deep_registry))

# ===== stage 3 =====
# Exact frozen structure lookup

def registry_row_for(architecture, method):
    if architecture == "shallow":
        frame = shallow_registry[
            (shallow_registry["method"] == method)
            & np.isclose(shallow_registry["target_flops"], TARGET_FLOPS)
        ]
    elif architecture == "deep":
        frame = deep_registry[
            (deep_registry["method"] == method)
            & (deep_registry["regime"] == "minimal")
            & np.isclose(deep_registry["target_flops"], TARGET_FLOPS)
        ]
    else:
        raise ValueError(architecture)
    if len(frame) != 1:
        raise RuntimeError(f"Expected one frozen row for {architecture}/{method}, found {len(frame)}")
    return frame.iloc[0]


def prune_map_from_csv(path):
    table = pd.read_csv(path)
    required = {"module_path", "channel_index"}
    if not required.issubset(table.columns):
        raise ValueError(f"Removed-group file lacks {sorted(required)}: {path}")
    return {
        str(layer): sorted(frame["channel_index"].astype(int).tolist())
        for layer, frame in table.groupby("module_path")
    }


def build_raw_student(architecture, method):
    row = registry_row_for(architecture, method)
    removed_path = REPO / str(row["removed_groups"])
    if not removed_path.exists():
        raise FileNotFoundError(removed_path)
    prune_map = prune_map_from_csv(removed_path)
    teacher = TEACHERS[architecture]
    student, surgery_audit = prune_cnn1d_channels(
        teacher,
        prune_map,
        example,
        minimum_remaining_per_layer=MINIMUM_WIDTH,
    )
    student = student.to(DEVICE)
    m0_flops = profile_forward_flops(teacher, example)["flops_per_item"]
    student_flops = profile_forward_flops(student, example)["flops_per_item"]
    realised = 1.0 - float(student_flops) / float(m0_flops)
    if abs(realised - float(row["realised_flops"])) > 0.02:
        raise RuntimeError(
            f"Realised-FLOP reconstruction mismatch for {architecture}/{method}: "
            f"rebuilt={realised:.4f}, registry={float(row['realised_flops']):.4f}"
        )
    return student, row, surgery_audit, realised

for architecture in RUN_ARCHITECTURES:
    for method in METHODS:
        model, row, _, realised = build_raw_student(architecture, method)
        print(architecture, method, "realised=", round(realised, 4), "params=", count_parameters(model))
        del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ===== stage 4 =====
# Class weights from the frozen training set.
if not hasattr(TRAIN_LOADER.dataset, "tensors"):
    raise TypeError("Notebook 25 expects the frozen TensorDataset bridge used by notebooks 19–22.")
train_y = TRAIN_LOADER.dataset.tensors[1].detach().cpu().numpy().astype(int)
counts = np.bincount(train_y, minlength=taxonomy.n_classes)
weights = np.zeros_like(counts, dtype=np.float64)
present = counts > 0
weights[present] = 1.0 / np.sqrt(counts[present])
weights[present] /= weights[present].mean()
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

# Fixed 10% membership; only shuffle order varies by recovery seed.
subset_gen = torch.Generator().manual_seed(0)
n_train = len(TRAIN_LOADER.dataset)
MINIMAL_INDICES = torch.randperm(n_train, generator=subset_gen)[: n_train // 10].tolist()


def make_recovery_loader(regime, recovery_seed):
    generator = torch.Generator().manual_seed(int(recovery_seed))
    if regime == "minimal_ce":
        dataset = Subset(TRAIN_LOADER.dataset, MINIMAL_INDICES)
        return DataLoader(
            dataset,
            batch_size=1024,
            shuffle=True,
            generator=generator,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )
    if regime in {"standard_ce", "full_ce"}:
        return DataLoader(
            TRAIN_LOADER.dataset,
            batch_size=int(getattr(TRAIN_LOADER, "batch_size", 4096) or 4096),
            shuffle=True,
            generator=generator,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )
    raise ValueError(regime)

print("Frozen minimal subset n:", len(MINIMAL_INDICES))
print("Class-weight range:", float(CLASS_WEIGHTS.min()), float(CLASS_WEIGHTS.max()))

# ===== stage 5 =====
@torch.no_grad()
def evaluate_model(model, architecture):
    logits, labels, _ = collect_logits(model, VAL_LOADER, device=DEVICE)
    audit = full_model_audit(logits, labels, taxonomy, DEFAULT_COST_PROFILES)
    awbir, _ = action_weighted_boundary_inversion_rate(
        TEACHER_LOGITS[architecture], logits, labels, robust_graph
    )
    audit["awbir"] = float(awbir)
    pred = logits.argmax(axis=1)
    teacher_pred = TEACHER_LOGITS[architecture].argmax(axis=1)
    audit.update(correction_degradation_metrics(labels, teacher_pred, pred))
    return logits, labels, audit


def correction_degradation_metrics(y_true, teacher_pred, student_pred):
    y = np.asarray(y_true, dtype=int)
    t = np.asarray(teacher_pred, dtype=int)
    s = np.asarray(student_pred, dtype=int)

    fine_t_correct = t == y
    fine_s_correct = s == y

    y_family = taxonomy.family_targets(y)
    t_family = taxonomy.family_targets(t)
    s_family = taxonomy.family_targets(s)
    family_t_correct = t_family == y_family
    family_s_correct = s_family == y_family

    y_binary = taxonomy.binary_targets(y)
    t_binary = taxonomy.binary_targets(t)
    s_binary = taxonomy.binary_targets(s)
    binary_t_correct = t_binary == y_binary
    binary_s_correct = s_binary == y_binary

    out = {}
    for level, tc, sc in [
        ("fine", fine_t_correct, fine_s_correct),
        ("family", family_t_correct, family_s_correct),
        ("binary", binary_t_correct, binary_s_correct),
    ]:
        correction = np.mean((~tc) & sc)
        degradation = np.mean(tc & (~sc))
        out[f"teacher_correction_{level}"] = float(correction)
        out[f"teacher_degradation_{level}"] = float(degradation)
        out[f"net_teacher_correction_{level}"] = float(correction - degradation)
    return out

# Dense-teacher audits.
teacher_rows = []
for architecture in RUN_ARCHITECTURES:
    _, _, audit = evaluate_model(TEACHERS[architecture], architecture)
    teacher_rows.append({"architecture": architecture, **audit})
teacher_audit = pd.DataFrame(teacher_rows)
teacher_audit.to_csv(OUT / "teacher_validation_audit.csv", index=False)
display(teacher_audit)

# ===== stage 6 =====
def seed_everything(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def recover_ce(raw_student, architecture, regime, recovery_seed):
    seed_everything(recovery_seed)
    model = deepcopy(raw_student).to(DEVICE)
    loader = make_recovery_loader(regime, recovery_seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

    if regime == "minimal_ce":
        max_epochs, patience, select_best = 1, None, False
    elif regime == "standard_ce":
        max_epochs, patience, select_best = 2, None, False
    elif regime == "full_ce":
        max_epochs, patience, select_best = 8, 3, True
    else:
        raise ValueError(regime)

    best_state = deepcopy(model.state_dict())
    best_f1 = -np.inf
    stale = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        for x, y in loader:
            x = x.float().to(DEVICE)
            y = y.long().to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.detach().cpu()) * len(y)
            seen += len(y)

        _, _, audit = evaluate_model(model, architecture)
        history.append({
            "epoch": epoch,
            "train_loss": running_loss / max(seen, 1),
            **{f"val_{k}": float(v) for k, v in audit.items() if np.isscalar(v)},
        })

        if select_best:
            current = float(audit["fine_macro_f1"])
            if current > best_f1 + 1e-6:
                best_f1 = current
                best_state = deepcopy(model.state_dict())
                stale = 0
            else:
                stale += 1
            if stale >= patience:
                break

    if select_best:
        model.load_state_dict(best_state)
    return model.eval(), pd.DataFrame(history)

# ===== stage 7 =====
RUNS_PATH = OUT / "recovery_seed_factorial_runs.csv"
HISTORY_PATH = OUT / "recovery_seed_factorial_epoch_history.csv"
RAW_PATH = OUT / "raw_structure_audit.csv"

run_rows = pd.read_csv(RUNS_PATH).to_dict("records") if RUNS_PATH.exists() else []
history_rows = pd.read_csv(HISTORY_PATH).to_dict("records") if HISTORY_PATH.exists() else []
raw_rows = pd.read_csv(RAW_PATH).to_dict("records") if RAW_PATH.exists() else []

done = {
    (str(r["architecture"]), str(r["method"]), str(r["regime"]), int(r["recovery_seed"]))
    for r in run_rows
}
raw_done = {(str(r["architecture"]), str(r["method"])) for r in raw_rows}

for architecture in RUN_ARCHITECTURES:
    for method in METHODS:
        raw_student, registry_row, surgery_audit, realised = build_raw_student(architecture, method)
        raw_student = raw_student.to(DEVICE).eval()
        if (architecture, method) not in raw_done:
            _, _, raw_audit = evaluate_model(raw_student, architecture)
            raw_rows.append({
                "architecture": architecture,
                "method": method,
                "target_flops": TARGET_FLOPS,
                "realised_flops": realised,
                "parameters": count_parameters(raw_student),
                "removed_groups": str(registry_row["removed_groups"]),
                **{k: float(v) for k, v in raw_audit.items() if np.isscalar(v)},
            })
            pd.DataFrame(raw_rows).to_csv(RAW_PATH, index=False)

        for regime in PROTOCOLS[architecture]:
            for recovery_seed in RECOVERY_SEEDS:
                key = (architecture, method, regime, int(recovery_seed))
                if key in done:
                    continue
                print(f"\n=== {architecture} | {method} | {regime} | seed={recovery_seed} ===")
                recovered, history = recover_ce(
                    raw_student, architecture, regime, recovery_seed
                )
                _, _, audit = evaluate_model(recovered, architecture)
                row = {
                    "architecture": architecture,
                    "method": method,
                    "regime": regime,
                    "recovery_seed": int(recovery_seed),
                    "target_flops": TARGET_FLOPS,
                    "realised_flops": realised,
                    "parameters": count_parameters(recovered),
                    "epochs_run": int(len(history)),
                    "removed_groups": str(registry_row["removed_groups"]),
                    **{k: float(v) for k, v in audit.items() if np.isscalar(v)},
                }
                run_rows.append(row)
                for record in history.to_dict("records"):
                    history_rows.append({
                        "architecture": architecture,
                        "method": method,
                        "regime": regime,
                        "recovery_seed": int(recovery_seed),
                        **record,
                    })
                pd.DataFrame(run_rows).to_csv(RUNS_PATH, index=False)
                pd.DataFrame(history_rows).to_csv(HISTORY_PATH, index=False)

                if SAVE_CHECKPOINTS:
                    checkpoint_dir = OUT / "checkpoints"
                    checkpoint_dir.mkdir(parents=True, exist_ok=True)
                    checkpoint = checkpoint_dir / f"{architecture}_{method}_{regime}_s{recovery_seed}.pt"
                    torch.save({
                        "state_dict": recovered.cpu().state_dict(),
                        "architecture": architecture,
                        "method": method,
                        "regime": regime,
                        "recovery_seed": int(recovery_seed),
                        "realised_flops": realised,
                        "class_names": CLASS_NAMES,
                    }, checkpoint)
                    recovered = recovered.to(DEVICE)

                print({
                    "awbir": row["awbir"],
                    "hsr": row["hsr_balanced_soc"],
                    "fine_f1": row["fine_macro_f1"],
                    "family_f1": row["family_macro_f1"],
                    "benign_to_attack": row["benign_to_attack_rate"],
                })
                del recovered
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        del raw_student
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

runs = pd.DataFrame(run_rows).sort_values(
    ["architecture", "regime", "recovery_seed", "method"]
)
raw = pd.DataFrame(raw_rows).sort_values(["architecture", "method"])
display(runs.tail(20))

# ===== stage 8 =====
METRICS = [
    "awbir",
    "hsr_balanced_soc",
    "fine_macro_f1",
    "family_macro_f1",
    "attack_to_benign_rate",
    "benign_to_attack_rate",
    "ece15",
    "nll",
    "brier",
    "teacher_correction_binary",
    "teacher_degradation_binary",
    "teacher_correction_family",
    "teacher_degradation_family",
    "teacher_correction_fine",
    "teacher_degradation_fine",
]

summary_rows = []
for keys, frame in runs.groupby(["architecture", "regime", "method"]):
    architecture, regime, method = keys
    for metric in METRICS:
        values = frame[metric].dropna().to_numpy(float)
        n = len(values)
        mean = float(np.mean(values)) if n else np.nan
        sd = float(np.std(values, ddof=1)) if n > 1 else np.nan
        if n > 1:
            half = float(student_t.ppf(0.975, n - 1) * sd / np.sqrt(n))
            lo, hi = mean - half, mean + half
        else:
            lo = hi = np.nan
        summary_rows.append({
            "architecture": architecture,
            "regime": regime,
            "method": method,
            "metric": metric,
            "n_seeds": n,
            "mean": mean,
            "sd": sd,
            "ci95_low": lo,
            "ci95_high": hi,
            "min": float(np.min(values)) if n else np.nan,
            "max": float(np.max(values)) if n else np.nan,
        })

method_summary = pd.DataFrame(summary_rows)
method_summary.to_csv(OUT / "method_seed_summary.csv", index=False)
display(method_summary[method_summary["metric"].isin(["awbir", "family_macro_f1", "benign_to_attack_rate"])])

# ===== stage 9 =====
RISK_TRANSFORMS = {
    "awbir": lambda x: x,
    "hsr_balanced_soc": lambda x: x,
    "benign_to_attack_rate": lambda x: x,
    "attack_to_benign_rate": lambda x: x,
    "fine_macro_f1": lambda x: 1.0 - x,
    "family_macro_f1": lambda x: 1.0 - x,
}

rei_rows = []
for (architecture, regime, recovery_seed), frame in runs.groupby(
    ["architecture", "regime", "recovery_seed"]
):
    raw_frame = raw[raw["architecture"] == architecture].set_index("method").loc[METHODS]
    rec_frame = frame.set_index("method").loc[METHODS]
    for metric, transform in RISK_TRANSFORMS.items():
        raw_risk = transform(raw_frame[metric].to_numpy(float))
        rec_risk = transform(rec_frame[metric].to_numpy(float))
        raw_spread = float(np.max(raw_risk) - np.min(raw_risk))
        rec_spread = float(np.max(rec_risk) - np.min(rec_risk))
        rei = 1.0 - rec_spread / raw_spread if raw_spread > 0 else np.nan
        rho = spearmanr(raw_risk, rec_risk).statistic
        rei_rows.append({
            "architecture": architecture,
            "regime": regime,
            "recovery_seed": int(recovery_seed),
            "metric": metric,
            "raw_spread": raw_spread,
            "recovered_spread": rec_spread,
            "recovery_equalisation_index": float(rei),
            "recovery_rank_retention": float(rho),
        })

rei_rrr = pd.DataFrame(rei_rows)
rei_rrr.to_csv(OUT / "recovery_equalisation_and_rank_retention.csv", index=False)
rei_summary = rei_rrr.groupby(["architecture", "regime", "metric"], as_index=False).agg(
    rei_mean=("recovery_equalisation_index", "mean"),
    rei_sd=("recovery_equalisation_index", "std"),
    rrr_mean=("recovery_rank_retention", "mean"),
    rrr_sd=("recovery_rank_retention", "std"),
)
rei_summary.to_csv(OUT / "recovery_equalisation_and_rank_retention_summary.csv", index=False)
display(rei_summary)

# ===== stage 10 =====
def two_way_no_replication(frame, metric):
    pivot = frame.pivot(index="method", columns="recovery_seed", values=metric).loc[METHODS, RECOVERY_SEEDS]
    x = pivot.to_numpy(float)
    grand = x.mean()
    method_means = x.mean(axis=1, keepdims=True)
    seed_means = x.mean(axis=0, keepdims=True)
    fitted = method_means + seed_means - grand
    residual = x - fitted
    ss_total = float(np.sum((x - grand) ** 2))
    ss_method = float(len(RECOVERY_SEEDS) * np.sum((method_means[:, 0] - grand) ** 2))
    ss_seed = float(len(METHODS) * np.sum((seed_means[0, :] - grand) ** 2))
    ss_interaction = float(np.sum(residual ** 2))
    denom = ss_total if ss_total > 0 else np.nan
    return {
        "ss_total": ss_total,
        "ss_method": ss_method,
        "ss_seed": ss_seed,
        "ss_method_x_seed_residual": ss_interaction,
        "fraction_method": ss_method / denom if np.isfinite(denom) else np.nan,
        "fraction_seed": ss_seed / denom if np.isfinite(denom) else np.nan,
        "fraction_method_x_seed_residual": ss_interaction / denom if np.isfinite(denom) else np.nan,
    }

variance_rows = []
for (architecture, regime), frame in runs.groupby(["architecture", "regime"]):
    for metric in RISK_TRANSFORMS:
        result = two_way_no_replication(frame, metric)
        variance_rows.append({
            "architecture": architecture,
            "regime": regime,
            "metric": metric,
            **result,
        })
variance = pd.DataFrame(variance_rows)
variance.to_csv(OUT / "variance_decomposition.csv", index=False)
display(variance)

# ===== stage 11 =====
LOWER_IS_BETTER = {
    "awbir": True,
    "hsr_balanced_soc": True,
    "benign_to_attack_rate": True,
    "attack_to_benign_rate": True,
    "fine_macro_f1": False,
    "family_macro_f1": False,
}

rank_rows = []
best_rows = []
for (architecture, regime), frame in runs.groupby(["architecture", "regime"]):
    for metric, lower in LOWER_IS_BETTER.items():
        rank_vectors = {}
        best_counts = {m: 0 for m in METHODS}
        for seed, sf in frame.groupby("recovery_seed"):
            values = sf.set_index("method").loc[METHODS, metric]
            ranks = values.rank(method="average", ascending=lower)
            rank_vectors[int(seed)] = ranks.to_numpy(float)
            best_value = values.min() if lower else values.max()
            for method in values.index[np.isclose(values.to_numpy(float), best_value)]:
                best_counts[str(method)] += 1
        seeds = sorted(rank_vectors)
        pairwise = []
        for i in range(len(seeds)):
            for j in range(i + 1, len(seeds)):
                pairwise.append(spearmanr(rank_vectors[seeds[i]], rank_vectors[seeds[j]]).statistic)
        rank_rows.append({
            "architecture": architecture,
            "regime": regime,
            "metric": metric,
            "mean_pairwise_rank_correlation": float(np.mean(pairwise)),
            "min_pairwise_rank_correlation": float(np.min(pairwise)),
            "max_pairwise_rank_correlation": float(np.max(pairwise)),
            "n_seed_pairs": len(pairwise),
        })
        for method, count in best_counts.items():
            best_rows.append({
                "architecture": architecture,
                "regime": regime,
                "metric": metric,
                "method": method,
                "best_count": count,
                "probability_best": count / len(RECOVERY_SEEDS),
            })

rank_stability = pd.DataFrame(rank_rows)
best_probability = pd.DataFrame(best_rows)
rank_stability.to_csv(OUT / "rank_stability.csv", index=False)
best_probability.to_csv(OUT / "probability_best.csv", index=False)
display(rank_stability)
display(best_probability[best_probability["metric"].isin(["awbir", "family_macro_f1"])])

# ===== stage 12 =====
# AWBIR recovery-seed traces.
for architecture in RUN_ARCHITECTURES:
    regimes = PROTOCOLS[architecture]
    fig, axes = plt.subplots(1, len(regimes), figsize=(7 * len(regimes), 5), squeeze=False)
    for ax, regime in zip(axes[0], regimes):
        frame = runs[(runs["architecture"] == architecture) & (runs["regime"] == regime)]
        for method in METHODS:
            mf = frame[frame["method"] == method].sort_values("recovery_seed")
            ax.plot(mf["recovery_seed"].astype(str), mf["awbir"], marker="o", label=method)
        ax.set_title(f"{architecture} / {regime}")
        ax.set_xlabel("Recovery seed")
        ax.set_ylabel("AWBIR (lower is better)")
        ax.grid(alpha=0.25)
    axes[0, 0].legend()
    fig.tight_layout()
    fig.savefig(OUT / f"{architecture}_awbir_by_recovery_seed.png", dpi=250)
    plt.show()

# Variance fractions.
plot_var = variance[variance["metric"].isin(["awbir", "hsr_balanced_soc", "family_macro_f1"])].copy()
plot_var["cell"] = plot_var["architecture"] + " / " + plot_var["regime"] + " / " + plot_var["metric"]
fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(plot_var))
for column, label in [
    ("fraction_method", "selection method"),
    ("fraction_seed", "recovery seed"),
    ("fraction_method_x_seed_residual", "method×seed / residual"),
]:
    values = plot_var[column].to_numpy(float)
    ax.bar(plot_var["cell"], values, bottom=bottom, label=label)
    bottom += values
ax.set_ylabel("Fraction of total sum of squares")
ax.set_xticklabels(plot_var["cell"], rotation=70, ha="right")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "variance_decomposition.png", dpi=250)
plt.show()

# REI vs RRR for AWBIR.
rr = rei_rrr[rei_rrr["metric"] == "awbir"]
fig, ax = plt.subplots(figsize=(8, 6))
for (architecture, regime), frame in rr.groupby(["architecture", "regime"]):
    ax.scatter(
        frame["recovery_equalisation_index"],
        frame["recovery_rank_retention"],
        label=f"{architecture}/{regime}",
        s=70,
    )
ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)
ax.set_xlabel("Recovery Equalisation Index")
ax.set_ylabel("Recovery Rank Retention (Spearman)")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "awbir_rei_vs_rrr.png", dpi=250)
plt.show()

# ===== stage 13 =====
awbir_rr = rei_rrr[rei_rrr["metric"] == "awbir"]
awbir_var = variance[variance["metric"] == "awbir"]

verdict_rows = []
for architecture in RUN_ARCHITECTURES:
    for regime in PROTOCOLS[architecture]:
        rr = awbir_rr[(awbir_rr["architecture"] == architecture) & (awbir_rr["regime"] == regime)]
        vr = awbir_var[(awbir_var["architecture"] == architecture) & (awbir_var["regime"] == regime)].iloc[0]
        verdict_rows.append({
            "architecture": architecture,
            "regime": regime,
            "mean_rei_awbir": float(rr["recovery_equalisation_index"].mean()),
            "mean_rrr_awbir": float(rr["recovery_rank_retention"].mean()),
            "method_variance_fraction_awbir": float(vr["fraction_method"]),
            "seed_variance_fraction_awbir": float(vr["fraction_seed"]),
            "interaction_residual_fraction_awbir": float(vr["fraction_method_x_seed_residual"]),
            "equalisation_descriptive": bool(rr["recovery_equalisation_index"].mean() >= 0.50),
            "rank_reordering_descriptive": bool(abs(rr["recovery_rank_retention"].mean()) <= 0.30),
            "recovery_variance_material": bool(
                vr["fraction_seed"] + vr["fraction_method_x_seed_residual"] >= vr["fraction_method"]
            ),
        })

verdict = {
    "analysis": "recovery_seed_factorial",
    "status": "descriptive_mechanism_confirmation_not_method_gate",
    "target_flops": TARGET_FLOPS,
    "methods": METHODS,
    "recovery_seeds": RECOVERY_SEEDS,
    "results": verdict_rows,
    "interpretation_rule": (
        "Recovery can be described as equalising or reordering selection only for the "
        "architecture/regime cells supported by the reported REI, RRR and variance components."
    ),
}
(OUT / "recovery_seed_factorial_verdict.json").write_text(
    json.dumps(verdict, indent=2), encoding="utf-8"
)

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_commit = None

artifacts = [
    RUNS_PATH,
    HISTORY_PATH,
    RAW_PATH,
    OUT / "method_seed_summary.csv",
    OUT / "recovery_equalisation_and_rank_retention.csv",
    OUT / "variance_decomposition.csv",
    OUT / "rank_stability.csv",
    OUT / "probability_best.csv",
    OUT / "recovery_seed_factorial_verdict.json",
]
manifest = {
    "notebook": "25_recovery_seed_factorial.ipynb",
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "git_commit": git_commit,
    "config_hash": config_hash(CFG),
    "target_flops": TARGET_FLOPS,
    "architectures": RUN_ARCHITECTURES,
    "methods": METHODS,
    "recovery_seeds": RECOVERY_SEEDS,
    "protocols": PROTOCOLS,
    "validation_only": True,
    "test_loader_used": False,
    "artifacts": [
        {
            "path": str(p.relative_to(REPO)),
            "sha256": file_sha256(p) if p.exists() else None,
        }
        for p in artifacts
    ],
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(verdict, indent=2))
print("Notebook 25 complete. Outputs:", OUT)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)